# Unit 7 / Chapter 7: Quantum Error Correction and Fault Tolerance for AI

> **Main Learning Objective:** Understand why today's noisy quantum computers cannot run the biggest quantum ML algorithms, what quantum error correction actually does, and when in the next 5 to 15 years fault tolerant quantum AI is expected to arrive.

| Section | Topic | Why it matters for AI |
|---|---|---|
| 7.1 | Noise in NISQ hardware | Sets the ceiling on today's quantum ML models |
| 7.2 | The bit flip and phase flip codes | The simplest error correction, and why they matter |
| 7.3 | Surface codes and logical qubits | The leading approach for large fault tolerant machines |
| 7.4 | What error correction costs | Overheads that determine when quantum AI becomes practical |

By the end of this unit you should be able to:

1. Explain in plain language why quantum error correction is much harder than classical error correction.
2. Describe the bit flip code and how it protects one logical qubit.
3. State roughly how many physical qubits are needed per logical qubit in the surface code.
4. Discuss why fault tolerance matters for quantum AI beyond the NISQ era.
5. Name at least two AI applications that only become plausible after fault tolerance arrives.

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100
print("Imports done.")

---
## Course check-in

This logs that you started **Unit 7**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 7
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 7.1: Noise in NISQ Hardware

Every gate on a real quantum computer has some chance of introducing an error. Modern hardware sits at roughly 0.1 percent to 1 percent error per two qubit gate. That sounds small until you multiply.

If a circuit has 1000 gates and each has a 0.1 percent error rate, the probability of running the whole circuit without any error is about (0.999)^1000 which is only 37 percent. At 1 percent error rate the probability drops to 4 in a million.

That is why NISQ era algorithms use shallow circuits (at most a few hundred gates) and short-lived quantum ML models. Quantum error correction is the way out.

In [ ]:
# Visualize how error rate and circuit depth combine to determine reliability.
depths = np.arange(1, 3001, 20)
for err, color in [(0.01, '#5B2C91'), (0.001, '#2A9D8F'), (0.0001, '#E76F51')]:
    reliability = (1 - err) ** depths
    plt.plot(depths, reliability, label=f'err/gate = {err}', color=color, lw=2)
plt.axhline(0.5, color='gray', ls='--', label='50% success')
plt.xlabel("circuit depth (number of gates)")
plt.ylabel("probability the whole circuit runs cleanly")
plt.title("Deeper circuits + noisy gates = quickly hopeless")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
# Section 7.2: The Bit Flip and Phase Flip Codes

Classical error correction repeats a bit three times: 0 becomes 000, 1 becomes 111. If one bit flips you can vote and recover.

Quantum error correction is harder for two reasons. First, you cannot copy an unknown quantum state (the no cloning theorem). Second, quantum errors come in two flavors: bit flips (X errors) and phase flips (Z errors). A useful code has to handle both.

The bit flip code encodes one logical qubit into three physical qubits:

$$ |0\rangle \rightarrow |000\rangle, \quad |1\rangle \rightarrow |111\rangle. $$

It uses a CNOT ladder to entangle the three qubits, then a majority vote at the end recovers the state. A separate "phase flip" code uses Hadamard sandwiching to protect against Z errors.

Combining both gives the Shor code, which uses 9 physical qubits per logical qubit and protects against any single qubit error.

In [ ]:
# Simulate the classical 3-vote bit flip protection to build intuition.
# We model it as: 3 copies, each has an independent chance of flipping. Vote.
rng = np.random.default_rng(1)
per_bit_err = np.linspace(0, 0.5, 40)
logical_err = []
for p in per_bit_err:
    # Probability of >=2 flips (which makes the vote wrong) with 3 iid bits
    p_vote_wrong = 3 * p**2 * (1-p) + p**3
    logical_err.append(p_vote_wrong)

plt.plot(per_bit_err, per_bit_err,   label='no protection (p)',            color='#E76F51', lw=2)
plt.plot(per_bit_err, logical_err,   label='3-bit majority vote', color='#2A9D8F', lw=2)
plt.plot([0, 0.5], [0.5, 0.5], color='gray', ls='--', alpha=0.5)
plt.axvline(0.5, color='gray', ls='--', alpha=0.5)
plt.xlabel("physical error rate p")
plt.ylabel("effective (logical) error rate")
plt.title("3-bit code helps IF the physical error rate is below the threshold")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Below p = 0.5 the code helps. Real quantum codes have their own threshold values.")

---
# Section 7.3: Surface Codes and Logical Qubits

The surface code is the most mature large scale quantum error correcting code. It uses a 2D grid of physical qubits and measures neighboring parities to detect errors. It is preferred because:

- It only needs nearest neighbor interactions (physically realistic).
- It has a threshold around 1 percent, forgiving compared to older codes.
- It scales smoothly: a d by d grid gives a code distance d and can tolerate roughly (d-1)/2 errors.

The cost is that one *logical* qubit requires many *physical* qubits. Estimates vary but typical figures are 1000 to 10 000 physical qubits per logical qubit for a code distance sufficient to run useful algorithms.

That is why headlines like "IBM has 1000 qubits" do not translate to 1000 useful qubits. You need to divide by 1000-ish.

In [ ]:
# Visualize the surface code physical vs logical qubit overhead.
distances = np.arange(3, 30, 2)   # odd distances
physical_per_logical = 2 * distances**2 - 1   # standard surface code lattice size

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(distances, physical_per_logical, marker='o', color='#5B2C91', lw=2)
ax.set_xlabel('code distance d')
ax.set_ylabel('physical qubits per logical qubit')
ax.set_title('Surface code overhead')
for d, n in zip(distances, physical_per_logical):
    if d in (3, 11, 21, 29):
        ax.annotate(f"d={d}: {n} qubits", (d, n), xytext=(6, -10),
                    textcoords='offset points', fontsize=9)
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

# What does an actual useful algorithm need?
print("For Shor's algorithm on 2048-bit RSA, estimates suggest ~2500 logical qubits,")
print("which at d=27 could mean 2500 * (2*27**2 - 1) = about 3.6 million physical qubits.")

---
# Section 7.4: What Error Correction Costs, and When it Arrives for AI

Fault tolerance is not free. Every logical gate has to be implemented through many physical gates. Common estimates say a logical gate might take 100 to 1000 physical gates.

For quantum AI, the key question is: which AI algorithms actually need fault tolerance?

| Algorithm | NISQ era? | Fault tolerance needed? | Rough timeline |
|---|---|---|---|
| VQC, QNN, quantum kernel (Unit 3) | Yes, small scale | Larger scale versions need FT | Today (small), 2030s (large) |
| VQE, QAOA (Unit 4) | Yes | Improved versions need FT | Today (toy), 2030s (industrial) |
| Grover on real databases | Marginally | Yes | 2030s |
| HHL on real linear systems | No | Yes | Late 2020s at earliest |
| Shor's algorithm | No | Yes | 2030s to 2040s |
| Full quantum ML on TB datasets | No | Definitely | Post 2035 |

## Activity 7: Predict the fault tolerance timeline

Estimate roughly when each of the following will be possible: a 100 logical qubit fault tolerant machine, a 1000 logical qubit one, and a 10 000 logical qubit one. Base your answer on the physical qubit growth rate you have heard about.

In [ ]:
# Guess a timeline: your predictions
my_prediction = {
    "100_logical_qubits_year":    2028,    # <- change to your own guess
    "1000_logical_qubits_year":   2032,
    "10000_logical_qubits_year":  2040,
    "reasoning":                  "..."
}

# For context, plot recent public physical qubit counts
years = np.array([2019, 2020, 2021, 2022, 2023, 2024, 2025])
ibm_physical = np.array([53, 65, 127, 433, 1121, 1121, 1121])   # rough public numbers

plt.plot(years, ibm_physical, marker='o', color='#5B2C91', lw=2, label='IBM physical qubits (rough)')
plt.axhline(1e6, color='gray', ls='--', label='1 million (est. for useful algorithms)')
plt.yscale('log')
plt.xlabel('year'); plt.ylabel('physical qubits (log)')
plt.title('Physical qubit growth so far')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()

print("Your prediction:", my_prediction)

---
# End-of-Unit Quiz

## Section A: Multiple choice

**1. Why is quantum error correction harder than classical error correction?**

A) Qubits are more fragile.
B) You cannot clone an unknown quantum state (no cloning theorem).
C) Physicists have not tried hard enough.
D) It requires more electricity.

<details><summary>Answer 1</summary>**B).** No cloning prevents naive replication; also, quantum errors come in more flavors (X and Z).</details>

---

**2. The bit flip code uses how many physical qubits to protect one logical qubit against a single bit flip?**

A) 1
B) 2
C) 3
D) 9

<details><summary>Answer 2</summary>**C) 3.** Majority vote across three copies recovers from one flip.</details>

---

**3. Roughly what is the surface code's error rate threshold?**

A) 10 percent
B) 1 percent
C) 0.001 percent
D) There is no threshold

<details><summary>Answer 3</summary>**B) about 1 percent.** Below that, adding more physical qubits reduces the logical error rate exponentially.</details>

---

**4. A typical estimate of physical qubits needed per logical qubit for useful fault tolerant algorithms is:**

A) 1
B) 10
C) 1000 to 10 000
D) Fewer than 5

<details><summary>Answer 4</summary>**C) 1000 to 10 000.** That is why current headline qubit counts do not translate directly into logical qubits.</details>

---

**5. Which of these algorithms is expected to NEED fault tolerance to be useful at scale?**

A) A shallow VQC on a small dataset
B) HHL on a large real world matrix
C) A tiny 4 qubit QNN demo
D) A single Bell state preparation

<details><summary>Answer 5</summary>**B) HHL on a large matrix.** HHL requires very deep circuits and precise arithmetic that noisy NISQ machines cannot deliver.</details>

## Section B: Short answer

**6. In your own words, why does compounding gate errors make deep quantum circuits impossible on NISQ hardware?**

<details><summary>Sample answer</summary>Each gate has some independent chance of introducing an error, so the probability of running the whole circuit cleanly decays roughly like (1 minus error rate)^depth, which becomes negligible for realistic error rates once you exceed a few hundred to a few thousand gates.</details>

---

**7. Give one reason surface codes are preferred over other quantum error correcting codes today.**

<details><summary>Sample answer</summary>They only require nearest neighbor interactions on a 2D grid, and they have a relatively high threshold around 1 percent, both of which match real hardware capabilities.</details>

---

**8. Explain in one sentence why "1000 physical qubits" on a company press release is not the same as "1000 useful qubits."**

<details><summary>Sample answer</summary>Because for fault tolerant computation each logical qubit consumes thousands of physical qubits for error correction, so 1000 physical qubits might correspond to zero or one useful logical qubit.</details>

---

**9. Name one quantum AI application from Units 2 to 6 that could run today (or nearly so) on NISQ hardware.**

<details><summary>Sample answer</summary>Any of: small VQC classifiers on 2 to 8 qubit data (Unit 3), tiny QAOA instances (Unit 4), small quantum kernel demos (Unit 3), or tiny VQE for small molecules (Unit 4). All benefit from shallow circuits and hybrid classical optimization.</details>

---

**10. (Bonus) In your own view, what is the single biggest hurdle between today's NISQ hardware and a practical fault tolerant machine capable of quantum ML at scale?**

<details><summary>Sample answer</summary>Any of: physical qubit count, gate fidelity, connectivity, cost of dilution refrigerators, software stack, or the sheer engineering complexity of integrating millions of components. A strong answer names one and explains why it dominates.</details>

---

## Unit 7 Summary

| Concept | Key idea |
|---|---|
| NISQ | Noisy, intermediate scale, quantum. Shallow circuits only, hybrid algorithms rule. |
| No cloning | You cannot copy an unknown quantum state, which makes error correction non trivial. |
| Bit flip code | 3 qubits protect 1 logical qubit against a single X error. |
| Phase flip code | Uses Hadamard sandwiching to protect against Z errors. |
| Shor code | Combines both, 9 qubits per logical qubit. |
| Surface code | Modern favorite: 2D grid, 1 percent threshold, nearest neighbor gates. |
| Overhead | Roughly 1000 to 10 000 physical qubits per logical qubit for useful algorithms. |
| AI implication | Small quantum ML is possible today; large scale waits for fault tolerance. |

---
## End-of-unit submission

Fill in your quiz answers and run this cell to submit.

In [ ]:
quiz_answers = {
    "q1": "",
    "q2": "",
    "q3": "",
    "q4": "",
    "q5": "",
    "q6_short_answer": "Type your answer to question 6 here.",
    "q7_short_answer": "Type your answer to question 7 here.",
    "q8_short_answer": "Type your answer to question 8 here.",
    "q9_short_answer": "Type your answer to question 9 here.",
    "q10_bonus":       "Type your bonus answer here, or leave blank."
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 7!")